# Feature testing notebook — profit with absolute values

Ce notebook part du `master.parquet` déjà produit par `Data_Processing.ipynb`, puis :
- redéfinit la cible avec la nouvelle formule  
  `profit_abs = |PR| - |C|`
- crée plusieurs familles de features
- entraîne un modèle simple
- compare des groupes de features
- évalue une logique de sélection **top-N par mois** compatible avec le challenge

> Hypothèse utilisée ici : le cutoff reste celui du cas MAG, donc la sélection pour `M+1` doit être faite avec l'information disponible au **7 du mois M**. Les sorties restent au niveau `(EID, MONTH, PEAKID)`. 


In [2]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

ROOT = Path.cwd()
BUILD_DIR = ROOT / "build_master_outputs"

MASTER_FILE_CANDIDATES = [
    BUILD_DIR / "master.parquet",
    ROOT / "master.parquet",
]

MASTER_FILE = None
for f in MASTER_FILE_CANDIDATES:
    if f.exists():
        MASTER_FILE = f
        break

if MASTER_FILE is None:
    raise FileNotFoundError(
        "master.parquet introuvable. Exécute d'abord Data_Processing.ipynb pour construire le dataset maître."
    )

print(f"Using master file: {MASTER_FILE}")


Using master file: /Users/mayaserine/Desktop/csd/CSD-challenge/build_master_outputs/master.parquet


In [3]:
df = pd.read_parquet(MASTER_FILE).copy()

# Nouvelle définition du profit demandée
df["profit_abs"] = df["PR"].abs() - df["C"].abs()
df["profitable_abs"] = (df["profit_abs"] > 0).astype(int)

# Colonnes de base utiles
df["year"] = df["MONTH"].str[:4].astype(int)
df["month_num"] = df["MONTH"].str[5:7].astype(int)
df["peak_label"] = df["PEAKID"].map({0: "OFF", 1: "ON"})

print(df.shape)
display(df[["EID", "MONTH", "PEAKID", "PR", "C", "profit_abs", "profitable_abs"]].head())
print("Positive rate (abs target):", round(df["profitable_abs"].mean(), 4))


(556630, 59)


,EID,MONTH,PEAKID,PR,C,profit_abs,profitable_abs
0,2,2020-01,0,0.0,0.0,0.0,0
1,2,2020-01,1,0.0,0.0,0.0,0
2,2,2020-02,0,0.0,0.0,0.0,0
3,2,2020-02,1,0.0,0.0,0.0,0
4,2,2020-03,0,0.0,0.0,0.0,0


Positive rate (abs target): 0.0276


In [4]:
def add_feature_blocks(data: pd.DataFrame) -> pd.DataFrame:
    out = data.copy()

    # --- simulated price strength ---
    monthly_psm_cols = [c for c in out.columns if c.startswith("PSM_sum_S")]
    daily_psd_cols = [c for c in out.columns if c.startswith("PSD_mean_S")]
    daily_psd_max_cols = [c for c in out.columns if c.startswith("PSD_max_S")]

    if monthly_psm_cols:
        out["PSM_sum_range"] = out[monthly_psm_cols].max(axis=1) - out[monthly_psm_cols].min(axis=1)
        out["PSM_sum_abs_mean"] = out[monthly_psm_cols].abs().mean(axis=1)
        out["PSM_sum_sign_agreement"] = (
            np.sign(out[monthly_psm_cols]).replace(0, np.nan).nunique(axis=1).fillna(0)
        )
    if daily_psd_cols:
        out["PSD_mean_range"] = out[daily_psd_cols].max(axis=1) - out[daily_psd_cols].min(axis=1)
        out["PSD_mean_abs_mean"] = out[daily_psd_cols].abs().mean(axis=1)
    if daily_psd_max_cols:
        out["PSD_max_abs_mean"] = out[daily_psd_max_cols].abs().mean(axis=1)

    # --- activation / dispersion ---
    for base in ["ACT_mean", "ACTD_mean"]:
        cols = [c for c in out.columns if c.startswith(base + "_S")]
        if cols:
            out[f"{base}_range"] = out[cols].max(axis=1) - out[cols].min(axis=1)
            out[f"{base}_abs_mean"] = out[cols].abs().mean(axis=1)

    # --- impact aggregates ---
    source_prefixes = ["WIND_mean", "SOLAR_mean", "HYDRO_mean", "NONRENWBAL_mean", "EXTERNAL_mean"]
    source_cols = [c for c in out.columns if any(c.startswith(p + "_S") for p in source_prefixes)]
    if source_cols:
        out["source_impact_abs_total"] = out[source_cols].abs().sum(axis=1)
        out["source_impact_signed_total"] = out[source_cols].sum(axis=1)

    load_cols = [c for c in out.columns if c.startswith("LOAD_mean_S")]
    outage_cols = [c for c in out.columns if c.startswith("TRANSOUTAGE_mean_S")]
    if load_cols:
        out["LOAD_abs_mean"] = out[load_cols].abs().mean(axis=1)
    if outage_cols:
        out["OUTAGE_abs_mean"] = out[outage_cols].abs().mean(axis=1)

    # --- interactions with PEAKID ---
    for col in ["PSM_sum_mean", "PSM_sum_std", "PSD_mean_mean", "PSD_mean_std", "ACT_mean_mean", "ACTD_mean_mean"]:
        if col in out.columns:
            out[f"{col}_x_peak"] = out[col] * out["PEAKID"]

    # --- calendar encoding ---
    out["month_sin"] = np.sin(2 * np.pi * out["month_num"] / 12)
    out["month_cos"] = np.cos(2 * np.pi * out["month_num"] / 12)

    return out

df_feat = add_feature_blocks(df)
print(df_feat.shape)


(556630, 81)


In [46]:
df_feat.columns

Index(['EID', 'MONTH', 'PEAKID', 'ACT_mean_S1', 'ACT_mean_S2', 'ACT_mean_S3', 'EXTERNAL_mean_S1', 'EXTERNAL_mean_S2', 'EXTERNAL_mean_S3', 'HYDRO_mean_S1',
       'HYDRO_mean_S2', 'HYDRO_mean_S3', 'LOAD_mean_S1', 'LOAD_mean_S2', 'LOAD_mean_S3', 'NONRENWBAL_mean_S1', 'NONRENWBAL_mean_S2', 'NONRENWBAL_mean_S3',
       'PSM_sum_S1', 'PSM_sum_S2', 'PSM_sum_S3', 'SOLAR_mean_S1', 'SOLAR_mean_S2', 'SOLAR_mean_S3', 'TRANSOUTAGE_mean_S1', 'TRANSOUTAGE_mean_S2',
       'TRANSOUTAGE_mean_S3', 'WIND_mean_S1', 'WIND_mean_S2', 'WIND_mean_S3', 'PSM_sum_mean', 'PSM_sum_std', 'ACT_mean_mean', 'ACT_mean_std', 'PSM_n_pos',
       'ACTD_mean_S1', 'ACTD_mean_S2', 'ACTD_mean_S3', 'PSD_max_S1', 'PSD_max_S2', 'PSD_max_S3', 'PSD_mean_S1', 'PSD_mean_S2', 'PSD_mean_S3', 'PSD_mean_mean',
       'PSD_mean_std', 'PSD_max_mean', 'PSD_max_std', 'ACTD_mean_mean', 'ACTD_mean_std', 'PR', 'C', 'profit', 'profitable', 'year', 'month_num', 'profit_abs',
       'profitable_abs', 'peak_label', 'PSM_sum_range', 'PSM_sum_abs_me

In [18]:
# Groupes de features à comparer
monthly_features = [
    c for c in df_feat.columns
    if c.startswith(("PSM_", "ACT_mean", "WIND_mean", "SOLAR_mean", "HYDRO_mean", "NONRENWBAL_mean", "EXTERNAL_mean", "LOAD_mean", "TRANSOUTAGE_mean"))
]

daily_features = [
    c for c in df_feat.columns
    if c.startswith(("PSD_", "ACTD_mean"))
]

engineered_features = [
    c for c in df_feat.columns
    if c.endswith(("_range", "_abs_mean", "_x_peak")) or c in [
        "source_impact_abs_total",
        "source_impact_signed_total",
        "month_sin",
        "month_cos",
        "PEAKID",
        "month_num",
    ]
]

# Enlever les colonnes cible / fuite / identifiants
forbidden = {"PR", "C", "profit", "profitable", "profit_abs", "profitable_abs", "EID", "MONTH", "peak_label"}
monthly_features = [c for c in monthly_features if c not in forbidden]
daily_features = [c for c in daily_features if c not in forbidden]
engineered_features = [c for c in engineered_features if c not in forbidden]

feature_sets = {
    "monthly_only": sorted(set(monthly_features + ["PEAKID", "month_num"])),
    "daily_only": sorted(set(daily_features + ["PEAKID", "month_num"])),
    "monthly_daily": sorted(set(monthly_features + daily_features + ["PEAKID", "month_num"])),
    "all_engineered": sorted(set(monthly_features + daily_features + engineered_features)),
}

{k: len(v) for k, v in feature_sets.items()}


{'monthly_only': 42,
 'daily_only': 25,
 'monthly_daily': 65,
 'all_engineered': 71}

In [17]:
def chronological_split(data, train_years=(2020, 2021, 2022), valid_years=(2023,)):
    train = data[data["year"].isin(train_years)].copy()
    valid = data[data["year"].isin(valid_years)].copy()
    return train, valid

train_df, valid_df = chronological_split(df_feat)

print("train:", train_df.shape, "valid:", valid_df.shape)
print("train positive rate:", round(train_df["profitable_abs"].mean(), 4))
print("valid positive rate:", round(valid_df["profitable_abs"].mean(), 4))


train: (391420, 81) valid: (165210, 81)
train positive rate: 0.0303
valid positive rate: 0.0212


In [7]:
def build_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        ("model", HistGradientBoostingClassifier(
            max_depth=6,
            learning_rate=0.05,
            max_iter=250,
            min_samples_leaf=50,
            random_state=42,
            class_weight="balanced",
        )),
    ])

def top_n_selection_eval(scored_df: pd.DataFrame, score_col="score", top_n=50, min_per_month=10, max_per_month=100):
    temp = scored_df.copy()
    temp = temp.sort_values(["MONTH", score_col], ascending=[True, False])

    picks = (
        temp.groupby("MONTH", group_keys=False)
            .head(top_n)
            .copy()
    )

    month_counts = picks.groupby("MONTH").size()
    if (month_counts < min_per_month).any():
        print("Warning: some months are below the required minimum.")
    if (month_counts > max_per_month).any():
        print("Warning: some months are above the required maximum.")

    y_true = picks["profitable_abs"]
    y_pred = np.ones(len(picks), dtype=int)

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = (
        picks["profitable_abs"].sum()
        / temp.groupby("MONTH")["profitable_abs"].sum().replace(0, np.nan).sum()
    )
    f1 = 0 if (precision + recall == 0) else 2 * precision * recall / (precision + recall)
    realized_profit_abs = picks["profit_abs"].sum()

    return {
        "selected_rows": len(picks),
        "avg_selected_per_month": round(len(picks) / picks["MONTH"].nunique(), 2),
        "precision_at_topn": precision,
        "pseudo_recall": float(recall) if pd.notna(recall) else np.nan,
        "pseudo_f1": f1,
        "profit_abs_sum_topn": realized_profit_abs,
    }, picks

def evaluate_feature_set(train_df, valid_df, features, name, top_n=50):
    model = build_model()
    X_train = train_df[features]
    y_train = train_df["profitable_abs"]
    X_valid = valid_df[features]
    y_valid = valid_df["profitable_abs"]

    model.fit(X_train, y_train)
    valid_scores = model.predict_proba(X_valid)[:, 1]

    row = {
        "feature_set": name,
        "n_features": len(features),
        "ap_valid": average_precision_score(y_valid, valid_scores),
    }

    if y_valid.nunique() > 1:
        row["roc_auc_valid"] = roc_auc_score(y_valid, valid_scores)
    else:
        row["roc_auc_valid"] = np.nan

    tmp = valid_df[["EID", "MONTH", "PEAKID", "profit_abs", "profitable_abs"]].copy()
    tmp["score"] = valid_scores

    top_metrics, picks = top_n_selection_eval(tmp, top_n=top_n)
    row.update(top_metrics)
    return row, model, picks


In [8]:
results = []
models = {}
picked_samples = {}

for name, feats in feature_sets.items():
    row, model, picks = evaluate_feature_set(train_df, valid_df, feats, name=name, top_n=50)
    results.append(row)
    models[name] = model
    picked_samples[name] = picks

results_df = pd.DataFrame(results).sort_values(
    ["profit_abs_sum_topn", "pseudo_f1", "ap_valid"], ascending=False
).reset_index(drop=True)

display(results_df)


,feature_set,n_features,ap_valid,roc_auc_valid,selected_rows,avg_selected_per_month,precision_at_topn,pseudo_recall,pseudo_f1,profit_abs_sum_topn
0,daily_only,25,0.389876,0.893652,600,50.0,0.585000,0.100000,0.170803,1.817528e+06
1,all_engineered,71,0.421982,0.916803,600,50.0,0.668333,0.114245,0.195134,1.667729e+06
2,monthly_daily,65,0.397665,0.915035,600,50.0,0.578333,0.098860,0.168856,1.546029e+06
3,monthly_only,42,0.165735,0.811053,600,50.0,0.383333,0.065527,0.111922,6.900882e+04


In [9]:
best_name = results_df.iloc[0]["feature_set"]
best_features = feature_sets[best_name]
best_model = models[best_name]

print("Best feature set:", best_name)
print("Number of features:", len(best_features))
print(best_features[:80])


Best feature set: daily_only
Number of features: 25
['ACTD_mean_S1', 'ACTD_mean_S2', 'ACTD_mean_S3', 'ACTD_mean_abs_mean', 'ACTD_mean_mean', 'ACTD_mean_mean_x_peak', 'ACTD_mean_range', 'ACTD_mean_std', 'PEAKID', 'PSD_max_S1', 'PSD_max_S2', 'PSD_max_S3', 'PSD_max_abs_mean', 'PSD_max_mean', 'PSD_max_std', 'PSD_mean_S1', 'PSD_mean_S2', 'PSD_mean_S3', 'PSD_mean_abs_mean', 'PSD_mean_mean', 'PSD_mean_mean_x_peak', 'PSD_mean_range', 'PSD_mean_std', 'PSD_mean_std_x_peak', 'month_num']


In [ ]:
# Importance approximative par permutation (version légère)
from sklearn.inspection import permutation_importance

X_valid_best = valid_df[best_features]
y_valid_best = valid_df["profitable_abs"]

perm = permutation_importance(
    best_model,
    X_valid_best,
    y_valid_best,
    n_repeats=5,
    random_state=42,
    scoring="average_precision",
)

imp_df = (
    pd.DataFrame({
        "feature": best_features,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

display(imp_df.head(25))


In [ ]:
# Export des opportunités top-N sur la validation
best_valid_scores = best_model.predict_proba(valid_df[best_features])[:, 1]

submission_like = valid_df[["MONTH", "PEAKID", "EID"]].copy()
submission_like["score"] = best_valid_scores
submission_like["PEAK_TYPE"] = submission_like["PEAKID"].map({0: "OFF", 1: "ON"})

topn = (
    submission_like
    .sort_values(["MONTH", "score"], ascending=[True, False])
    .groupby("MONTH", group_keys=False)
    .head(50)
    .loc[:, ["MONTH", "PEAK_TYPE", "EID", "score"]]
    .rename(columns={"MONTH": "TARGET_MONTH"})
    .reset_index(drop=True)
)

out_file = ROOT / "opportunities_abs_top50_validation.csv"
topn.to_csv(out_file, index=False)

print(f"Saved: {out_file}")
display(topn.head(20))


In [21]:
# ============================================
# REGRESSION SUR profit_abs
# Objectif: prédire la valeur du profit positif
# puis classer les opportunités par score
# ============================================

import numpy as np
import pandas as pd

from sklearn.experimental import enable_hist_gradient_boosting  # sometimes not needed depending on sklearn version
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --------------------------------------------------
# 1) Définir la cible de régression
# --------------------------------------------------
# On garde seulement la partie positive du profit comme cible business
df_model = df_feat.copy()
df_model["profit_target_reg"] = np.maximum(df_model["profit_abs"], 0)

# --------------------------------------------------
# 2) Choisir le feature set à tester
# --------------------------------------------------
# Remplace par celui que tu veux comparer :
# "daily_only"
# "monthly_only"
# "monthly_daily"
# "all_engineered"

feature_cols = feature_sets["daily_only"]   # <<< change ici si tu veux tester un autre set

# garder seulement les colonnes disponibles
feature_cols = [c for c in feature_cols if c in df_model.columns]

print(f"Nombre de features utilisées: {len(feature_cols)}")

# --------------------------------------------------
# 3) Train / validation split (chronologique)
# --------------------------------------------------
# Suppose que train_df et valid_df existent déjà dans ton notebook.
# Sinon, adapte avec ton split temporel habituel.

train_reg = train_df.copy()
valid_reg = valid_df.copy()

X_train = train_reg[feature_cols].copy()
X_valid = valid_reg[feature_cols].copy()

y_train = np.maximum(train_reg["profit_abs"], 0)
y_valid = np.maximum(valid_reg["profit_abs"], 0)

# remplir NA si nécessaire
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_valid = X_valid.replace([np.inf, -np.inf], np.nan).fillna(0)

# --------------------------------------------------
# 4) Entraîner le modèle
# --------------------------------------------------
reg_model = HistGradientBoostingRegressor(
    loss="squared_error",
    learning_rate=0.05,
    max_iter=300,
    max_depth=6,
    min_samples_leaf=20,
    l2_regularization=1.0,
    random_state=42
)

reg_model.fit(X_train, y_train)

# --------------------------------------------------
# 5) Prédictions
# --------------------------------------------------
train_reg = train_reg.copy()
valid_reg = valid_reg.copy()

train_reg["pred_profit_reg"] = reg_model.predict(X_train)
valid_reg["pred_profit_reg"] = reg_model.predict(X_valid)

# par sécurité : on clippe à 0
train_reg["pred_profit_reg"] = train_reg["pred_profit_reg"].clip(lower=0)
valid_reg["pred_profit_reg"] = valid_reg["pred_profit_reg"].clip(lower=0)

# --------------------------------------------------
# 6) Métriques de régression classiques
# --------------------------------------------------
mae_valid = mean_absolute_error(y_valid, valid_reg["pred_profit_reg"])
rmse_valid = np.sqrt(mean_squared_error(y_valid, valid_reg["pred_profit_reg"]))
r2_valid = r2_score(y_valid, valid_reg["pred_profit_reg"])

print("=== Regression metrics on validation ===")
print(f"MAE  : {mae_valid:,.4f}")
print(f"RMSE : {rmse_valid:,.4f}")
print(f"R2   : {r2_valid:,.4f}")

# --------------------------------------------------
# 7) Évaluation business : top 50 par mois
# --------------------------------------------------
# On trie par profit prédit, puis on prend top 50 / mois
# Adapte le nom de colonne du mois si besoin : "MONTH"
TOP_N = 50
MONTH_COL = "MONTH"

valid_topn = (
    valid_reg
    .sort_values([MONTH_COL, "pred_profit_reg"], ascending=[True, False])
    .groupby(MONTH_COL, group_keys=False)
    .head(TOP_N)
    .copy()
)

# profit réellement capturé
profit_abs_sum_topn_reg = valid_topn["profit_abs"].sum()

# combien de lignes réellement profitables dans le topN
valid_topn["is_positive_profit_abs"] = (valid_topn["profit_abs"] > 0).astype(int)
precision_at_topn_reg = valid_topn["is_positive_profit_abs"].mean()

# pseudo recall : nb de vrais positifs capturés / nb total de positifs validation
total_positive_valid = (valid_reg["profit_abs"] > 0).sum()
captured_positive_valid = valid_topn["is_positive_profit_abs"].sum()
pseudo_recall_reg = captured_positive_valid / total_positive_valid if total_positive_valid > 0 else 0.0

pseudo_f1_reg = (
    2 * precision_at_topn_reg * pseudo_recall_reg / (precision_at_topn_reg + pseudo_recall_reg)
    if (precision_at_topn_reg + pseudo_recall_reg) > 0 else 0.0
)

# moyenne du topN par mois
avg_selected_per_month_reg = valid_topn.groupby(MONTH_COL).size().mean()

print("\n=== Business metrics on validation (top 50 / month) ===")
print(f"Selected rows            : {len(valid_topn)}")
print(f"Avg selected / month     : {avg_selected_per_month_reg:.1f}")
print(f"Precision@TopN           : {precision_at_topn_reg:.6f}")
print(f"Pseudo Recall            : {pseudo_recall_reg:.6f}")
print(f"Pseudo F1                : {pseudo_f1_reg:.6f}")
print(f"Profit abs sum TopN      : {profit_abs_sum_topn_reg:,.2f}")

# --------------------------------------------------
# 8) Sauvegarde des opportunités sélectionnées
# --------------------------------------------------
cols_to_export = [
    c for c in [
        "EID", "MONTH", "PEAKID",
        "profit_abs", "pred_profit_reg"
    ] if c in valid_topn.columns
]

valid_topn[cols_to_export].to_csv("opportunities_regression_top50_validation.csv", index=False)
print("\nFichier exporté : opportunities_regression_top50_validation.csv")

# --------------------------------------------------
# 9) Résumé final dans un DataFrame
# --------------------------------------------------
regression_summary = pd.DataFrame([{
    "model_type": "HistGradientBoostingRegressor",
    "feature_set": "daily_only",
    "n_features": len(feature_cols),
    "mae_valid": mae_valid,
    "rmse_valid": rmse_valid,
    "r2_valid": r2_valid,
    "selected_rows": len(valid_topn),
    "avg_selected_per_month": avg_selected_per_month_reg,
    "precision_at_topn": precision_at_topn_reg,
    "pseudo_recall": pseudo_recall_reg,
    "pseudo_f1": pseudo_f1_reg,
    "profit_abs_sum_topn": profit_abs_sum_topn_reg
}])

display(regression_summary)

Nombre de features utilisées: 25
=== Regression metrics on validation ===
MAE  : 84.4559
RMSE : 1,256.9052
R2   : 0.1273

=== Business metrics on validation (top 50 / month) ===
Selected rows            : 600
Avg selected / month     : 50.0
Precision@TopN           : 0.556667
Pseudo Recall            : 0.095157
Pseudo F1                : 0.162530
Profit abs sum TopN      : 2,249,721.08

Fichier exporté : opportunities_regression_top50_validation.csv


,model_type,feature_set,n_features,mae_valid,rmse_valid,r2_valid,selected_rows,avg_selected_per_month,precision_at_topn,pseudo_recall,pseudo_f1,profit_abs_sum_topn
0,HistGradientBoostingRegressor,daily_only,25,84.455925,1256.905214,0.127333,600,50.0,0.556667,0.095157,0.16253,2.249721e+06


In [22]:
# ============================================
# REGRESSION SUR profit_abs
# Objectif: prédire la valeur du profit positif
# puis classer les opportunités par score
# ============================================

import numpy as np
import pandas as pd

from sklearn.experimental import enable_hist_gradient_boosting  # sometimes not needed depending on sklearn version
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --------------------------------------------------
# 1) Définir la cible de régression
# --------------------------------------------------
# On garde seulement la partie positive du profit comme cible business
df_model = df_feat.copy()
df_model["profit_target_reg"] = np.maximum(df_model["profit_abs"], 0)

# --------------------------------------------------
# 2) Choisir le feature set à tester
# --------------------------------------------------
# Remplace par celui que tu veux comparer :
# "daily_only"
# "monthly_only"
# "monthly_daily"
# "all_engineered"

feature_cols = feature_sets["all_engineered"]   # <<< change ici si tu veux tester un autre set

# garder seulement les colonnes disponibles
feature_cols = [c for c in feature_cols if c in df_model.columns]

print(f"Nombre de features utilisées: {len(feature_cols)}")

# --------------------------------------------------
# 3) Train / validation split (chronologique)
# --------------------------------------------------
# Suppose que train_df et valid_df existent déjà dans ton notebook.
# Sinon, adapte avec ton split temporel habituel.

train_reg = train_df.copy()
valid_reg = valid_df.copy()

X_train = train_reg[feature_cols].copy()
X_valid = valid_reg[feature_cols].copy()

y_train = np.maximum(train_reg["profit_abs"], 0)
y_valid = np.maximum(valid_reg["profit_abs"], 0)

# remplir NA si nécessaire
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_valid = X_valid.replace([np.inf, -np.inf], np.nan).fillna(0)

# --------------------------------------------------
# 4) Entraîner le modèle
# --------------------------------------------------
reg_model = HistGradientBoostingRegressor(
    loss="squared_error",
    learning_rate=0.05,
    max_iter=300,
    max_depth=6,
    min_samples_leaf=20,
    l2_regularization=1.0,
    random_state=42
)

reg_model.fit(X_train, y_train)

# --------------------------------------------------
# 5) Prédictions
# --------------------------------------------------
train_reg = train_reg.copy()
valid_reg = valid_reg.copy()

train_reg["pred_profit_reg"] = reg_model.predict(X_train)
valid_reg["pred_profit_reg"] = reg_model.predict(X_valid)

# par sécurité : on clippe à 0
train_reg["pred_profit_reg"] = train_reg["pred_profit_reg"].clip(lower=0)
valid_reg["pred_profit_reg"] = valid_reg["pred_profit_reg"].clip(lower=0)

# --------------------------------------------------
# 6) Métriques de régression classiques
# --------------------------------------------------
mae_valid = mean_absolute_error(y_valid, valid_reg["pred_profit_reg"])
rmse_valid = np.sqrt(mean_squared_error(y_valid, valid_reg["pred_profit_reg"]))
r2_valid = r2_score(y_valid, valid_reg["pred_profit_reg"])

print("=== Regression metrics on validation ===")
print(f"MAE  : {mae_valid:,.4f}")
print(f"RMSE : {rmse_valid:,.4f}")
print(f"R2   : {r2_valid:,.4f}")

# --------------------------------------------------
# 7) Évaluation business : top 50 par mois
# --------------------------------------------------
# On trie par profit prédit, puis on prend top 50 / mois
# Adapte le nom de colonne du mois si besoin : "MONTH"
TOP_N = 50
MONTH_COL = "MONTH"

valid_topn = (
    valid_reg
    .sort_values([MONTH_COL, "pred_profit_reg"], ascending=[True, False])
    .groupby(MONTH_COL, group_keys=False)
    .head(TOP_N)
    .copy()
)

# profit réellement capturé
profit_abs_sum_topn_reg = valid_topn["profit_abs"].sum()

# combien de lignes réellement profitables dans le topN
valid_topn["is_positive_profit_abs"] = (valid_topn["profit_abs"] > 0).astype(int)
precision_at_topn_reg = valid_topn["is_positive_profit_abs"].mean()

# pseudo recall : nb de vrais positifs capturés / nb total de positifs validation
total_positive_valid = (valid_reg["profit_abs"] > 0).sum()
captured_positive_valid = valid_topn["is_positive_profit_abs"].sum()
pseudo_recall_reg = captured_positive_valid / total_positive_valid if total_positive_valid > 0 else 0.0

pseudo_f1_reg = (
    2 * precision_at_topn_reg * pseudo_recall_reg / (precision_at_topn_reg + pseudo_recall_reg)
    if (precision_at_topn_reg + pseudo_recall_reg) > 0 else 0.0
)

# moyenne du topN par mois
avg_selected_per_month_reg = valid_topn.groupby(MONTH_COL).size().mean()

print("\n=== Business metrics on validation (top 50 / month) ===")
print(f"Selected rows            : {len(valid_topn)}")
print(f"Avg selected / month     : {avg_selected_per_month_reg:.1f}")
print(f"Precision@TopN           : {precision_at_topn_reg:.6f}")
print(f"Pseudo Recall            : {pseudo_recall_reg:.6f}")
print(f"Pseudo F1                : {pseudo_f1_reg:.6f}")
print(f"Profit abs sum TopN      : {profit_abs_sum_topn_reg:,.2f}")

# --------------------------------------------------
# 8) Sauvegarde des opportunités sélectionnées
# --------------------------------------------------
cols_to_export = [
    c for c in [
        "EID", "MONTH", "PEAKID",
        "profit_abs", "pred_profit_reg"
    ] if c in valid_topn.columns
]

valid_topn[cols_to_export].to_csv("opportunities_regression_top50_validation.csv", index=False)
print("\nFichier exporté : opportunities_regression_top50_validation.csv")

# --------------------------------------------------
# 9) Résumé final dans un DataFrame
# --------------------------------------------------
regression_summary = pd.DataFrame([{
    "model_type": "HistGradientBoostingRegressor",
    "feature_set": "daily_only",
    "n_features": len(feature_cols),
    "mae_valid": mae_valid,
    "rmse_valid": rmse_valid,
    "r2_valid": r2_valid,
    "selected_rows": len(valid_topn),
    "avg_selected_per_month": avg_selected_per_month_reg,
    "precision_at_topn": precision_at_topn_reg,
    "pseudo_recall": pseudo_recall_reg,
    "pseudo_f1": pseudo_f1_reg,
    "profit_abs_sum_topn": profit_abs_sum_topn_reg
}])

display(regression_summary)

Nombre de features utilisées: 71
=== Regression metrics on validation ===
MAE  : 100.9964
RMSE : 1,425.9834
R2   : -0.1232

=== Business metrics on validation (top 50 / month) ===
Selected rows            : 600
Avg selected / month     : 50.0
Precision@TopN           : 0.545000
Pseudo Recall            : 0.093162
Pseudo F1                : 0.159124
Profit abs sum TopN      : 2,207,336.81

Fichier exporté : opportunities_regression_top50_validation.csv


,model_type,feature_set,n_features,mae_valid,rmse_valid,r2_valid,selected_rows,avg_selected_per_month,precision_at_topn,pseudo_recall,pseudo_f1,profit_abs_sum_topn
0,HistGradientBoostingRegressor,daily_only,71,100.996443,1425.983354,-0.123239,600,50.0,0.545,0.093162,0.159124,2.207337e+06


Les modèles de régression alignés sur la cible profit_abs améliorent la performance business par rapport à la classification. Le meilleur résultat est obtenu avec les features daily_only, qui capturent ~2.25M de profit dans le top-50 mensuel, contre ~1.8M pour la classification. L’ajout de features engineered supplémentaires n’améliore pas ce résultat.

In [24]:
# ============================================
# REGRESSION SUR profit_abs
# Objectif: prédire la valeur du profit positif
# puis classer les opportunités par score
# ============================================

import numpy as np
import pandas as pd

from sklearn.experimental import enable_hist_gradient_boosting  # sometimes not needed depending on sklearn version
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --------------------------------------------------
# 1) Définir la cible de régression
# --------------------------------------------------
# On garde seulement la partie positive du profit comme cible business
df_model = df_feat.copy()
df_model["profit_target_reg"] = np.maximum(df_model["profit_abs"], 0)

# --------------------------------------------------
# 2) Choisir le feature set à tester
# --------------------------------------------------
# Remplace par celui que tu veux comparer :
# "daily_only"
# "monthly_only"
# "monthly_daily"
# "all_engineered"

feature_cols = feature_sets["daily_only"]   # <<< change ici si tu veux tester un autre set

# garder seulement les colonnes disponibles
feature_cols = [c for c in feature_cols if c in df_model.columns]

print(f"Nombre de features utilisées: {len(feature_cols)}")

# --------------------------------------------------
# 3) Train / validation split (chronologique)
# --------------------------------------------------
# Suppose que train_df et valid_df existent déjà dans ton notebook.
# Sinon, adapte avec ton split temporel habituel.

train_reg = train_df.copy()
valid_reg = valid_df.copy()

X_train = train_reg[feature_cols].copy()
X_valid = valid_reg[feature_cols].copy()

y_train = np.log1p(np.maximum(train_reg["profit_abs"], 0))
y_valid = np.log1p(np.maximum(valid_reg["profit_abs"], 0))

# remplir NA si nécessaire
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_valid = X_valid.replace([np.inf, -np.inf], np.nan).fillna(0)

# --------------------------------------------------
# 4) Entraîner le modèle
# --------------------------------------------------
reg_model = HistGradientBoostingRegressor(
    loss="squared_error",
    learning_rate=0.05,
    max_iter=300,
    max_depth=6,
    min_samples_leaf=20,
    l2_regularization=1.0,
    random_state=42
)

reg_model.fit(X_train, y_train)

# --------------------------------------------------
# 5) Prédictions
# --------------------------------------------------
train_reg = train_reg.copy()
valid_reg = valid_reg.copy()

train_reg["pred_profit_reg"] = reg_model.predict(X_train)
valid_reg["pred_profit_reg"] = np.expm1(reg_model.predict(X_valid))
# par sécurité : on clippe à 0
train_reg["pred_profit_reg"] = train_reg["pred_profit_reg"].clip(lower=0)
valid_reg["pred_profit_reg"] = valid_reg["pred_profit_reg"].clip(lower=0)

# --------------------------------------------------
# 6) Métriques de régression classiques
# --------------------------------------------------
mae_valid = mean_absolute_error(y_valid, valid_reg["pred_profit_reg"])
rmse_valid = np.sqrt(mean_squared_error(y_valid, valid_reg["pred_profit_reg"]))
r2_valid = r2_score(y_valid, valid_reg["pred_profit_reg"])

print("=== Regression metrics on validation ===")
print(f"MAE  : {mae_valid:,.4f}")
print(f"RMSE : {rmse_valid:,.4f}")
print(f"R2   : {r2_valid:,.4f}")

# --------------------------------------------------
# 7) Évaluation business : top 50 par mois
# --------------------------------------------------
# On trie par profit prédit, puis on prend top 50 / mois
# Adapte le nom de colonne du mois si besoin : "MONTH"
TOP_N = 50
MONTH_COL = "MONTH"

valid_topn = (
    valid_reg
    .sort_values([MONTH_COL, "pred_profit_reg"], ascending=[True, False])
    .groupby(MONTH_COL, group_keys=False)
    .head(TOP_N)
    .copy()
)

# profit réellement capturé
profit_abs_sum_topn_reg = valid_topn["profit_abs"].sum()

# combien de lignes réellement profitables dans le topN
valid_topn["is_positive_profit_abs"] = (valid_topn["profit_abs"] > 0).astype(int)
precision_at_topn_reg = valid_topn["is_positive_profit_abs"].mean()

# pseudo recall : nb de vrais positifs capturés / nb total de positifs validation
total_positive_valid = (valid_reg["profit_abs"] > 0).sum()
captured_positive_valid = valid_topn["is_positive_profit_abs"].sum()
pseudo_recall_reg = captured_positive_valid / total_positive_valid if total_positive_valid > 0 else 0.0

pseudo_f1_reg = (
    2 * precision_at_topn_reg * pseudo_recall_reg / (precision_at_topn_reg + pseudo_recall_reg)
    if (precision_at_topn_reg + pseudo_recall_reg) > 0 else 0.0
)

# moyenne du topN par mois
avg_selected_per_month_reg = valid_topn.groupby(MONTH_COL).size().mean()

print("\n=== Business metrics on validation (top 50 / month) ===")
print(f"Selected rows            : {len(valid_topn)}")
print(f"Avg selected / month     : {avg_selected_per_month_reg:.1f}")
print(f"Precision@TopN           : {precision_at_topn_reg:.6f}")
print(f"Pseudo Recall            : {pseudo_recall_reg:.6f}")
print(f"Pseudo F1                : {pseudo_f1_reg:.6f}")
print(f"Profit abs sum TopN      : {profit_abs_sum_topn_reg:,.2f}")

# --------------------------------------------------
# 8) Sauvegarde des opportunités sélectionnées
# --------------------------------------------------
cols_to_export = [
    c for c in [
        "EID", "MONTH", "PEAKID",
        "profit_abs", "pred_profit_reg"
    ] if c in valid_topn.columns
]

valid_topn[cols_to_export].to_csv("opportunities_regression_top50_validation.csv", index=False)
print("\nFichier exporté : opportunities_regression_top50_validation.csv")

# --------------------------------------------------
# 9) Résumé final dans un DataFrame
# --------------------------------------------------
regression_summary = pd.DataFrame([{
    "model_type": "HistGradientBoostingRegressor",
    "feature_set": "daily_only",
    "n_features": len(feature_cols),
    "mae_valid": mae_valid,
    "rmse_valid": rmse_valid,
    "r2_valid": r2_valid,
    "selected_rows": len(valid_topn),
    "avg_selected_per_month": avg_selected_per_month_reg,
    "precision_at_topn": precision_at_topn_reg,
    "pseudo_recall": pseudo_recall_reg,
    "pseudo_f1": pseudo_f1_reg,
    "profit_abs_sum_topn": profit_abs_sum_topn_reg
}])

display(regression_summary)

Nombre de features utilisées: 25
=== Regression metrics on validation ===
MAE  : 4.3227
RMSE : 60.6455
R2   : -4,684.7524

=== Business metrics on validation (top 50 / month) ===
Selected rows            : 600
Avg selected / month     : 50.0
Precision@TopN           : 0.578333
Pseudo Recall            : 0.098860
Pseudo F1                : 0.168856
Profit abs sum TopN      : 2,216,960.16

Fichier exporté : opportunities_regression_top50_validation.csv


,model_type,feature_set,n_features,mae_valid,rmse_valid,r2_valid,selected_rows,avg_selected_per_month,precision_at_topn,pseudo_recall,pseudo_f1,profit_abs_sum_topn
0,HistGradientBoostingRegressor,daily_only,25,4.32268,60.645496,-4684.752372,600,50.0,0.578333,0.09886,0.168856,2.216960e+06


## Comment l'utiliser

1. Exécute d'abord `Data_Processing.ipynb`.
2. Ouvre ce notebook.
3. Lance toutes les cellules.
4. Regarde `results_df` pour voir quel bloc de features fonctionne le mieux.
5. Ajuste ensuite :
   - les années de train / validation
   - le `top_n`
   - les features dérivées
   - les hyperparamètres du modèle

## Idées de tests ensuite

- comparer `top_n = 20, 30, 50, 75, 100`
- entraîner un modèle séparé pour `PEAKID = 0` et `PEAKID = 1`
- prédire directement `profit_abs` avec un modèle de régression
- ajouter des features historiques par EID :
  - fréquence passée d'opportunités profitables
  - moyenne glissante du coût
  - moyenne glissante du prix réalisé
  - volatilité passée
- comparer la cible binaire `profitable_abs` à une cible de ranking basée sur `profit_abs`
